In [0]:

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, lit



#Duplicate detection
def check_duplicates(df: DataFrame, key_cols: list, table_name: str = "") -> dict:
    total_rows = df.count()
    distinct_keys = df.select(*key_cols).distinct().count()
    duplicate_rows = total_rows - distinct_keys

    result = {
        "table": table_name,
        "check": "duplicity",
        "total_rows": total_rows,
        "distinct_keys": distinct_keys,
        "duplicate_rows": duplicate_rows,
        "duplicate_pct": round((duplicate_rows / total_rows * 100), 2) if total_rows > 0 else 0.0,
        "status": "OK" if duplicate_rows == 0 else "WARNING",
    }
    return result


#Missing value detection
def check_missing_values(df: DataFrame, table_name: str = "") -> list:
    total_rows = df.count()
    results = []

    for field in df.schema.fields:
        column_name = field.name
        dtype = str(field.dataType)

        if "DoubleType" in dtype or "FloatType" in dtype:
            null_count = df.filter(col(column_name).isNull() | isnan(col(column_name))).count()
        else:
            null_count = df.filter(col(column_name).isNull()).count()

        null_pct = round((null_count / total_rows * 100), 2) if total_rows > 0 else 0.0

        results.append({
            "table": table_name,
            "check": "missing_values",
            "column": column_name,
            "null_count": null_count,
            "null_pct": null_pct,
            "status": "OK" if null_pct == 0 else ("WARNING" if null_pct < 5 else "CRITICAL"),
        })

    return results



#Foreign key validation
def validate_foreign_key(child_df: DataFrame, parent_df: DataFrame, key_col: str,
                          child_table: str = "", parent_table: str = "") -> dict:
    
    child_keys = child_df.select(key_col).distinct()
    parent_keys = parent_df.select(key_col).distinct()

    orphan_keys = child_keys.join(parent_keys, on=key_col, how="left_anti")
    orphan_count = orphan_keys.count()
    total_keys = child_keys.count()

    result = {
        "child_table": child_table,
        "parent_table": parent_table,
        "check": "foreign_key",
        "key_column": key_col,
        "total_distinct_keys": total_keys,
        "orphan_keys": orphan_count,
        "orphan_pct": round((orphan_count / total_keys * 100), 2) if total_keys > 0 else 0.0,
        "status": "OK" if orphan_count == 0 else "WARNING",
    }
    return result



#Cross-table consistency check
def check_consistency(df_a: DataFrame, df_b: DataFrame, join_cols: list,
                       compare_col_a: str, compare_col_b: str,
                       tolerance: float = 1.0, table_a: str = "", table_b: str = "") -> dict:

    joined_df = df_a.select(*join_cols, col(compare_col_a).alias("valor_a")) \
        .join(df_b.select(*join_cols, col(compare_col_b).alias("valor_b")), on=join_cols, how="inner")

    joined_df = joined_df.withColumn("diferenca_absoluta", (col("valor_a") - col("valor_b")))
    total_compared = joined_df.count()
    inconsistent_count = joined_df.filter(
        (col("diferenca_absoluta") > tolerance) | (col("diferenca_absoluta") < -tolerance)
    ).count()

    result = {
        "table_a": table_a,
        "table_b": table_b,
        "check": "consistency",
        "compared_columns": f"{compare_col_a} vs {compare_col_b}",
        "total_compared": total_compared,
        "inconsistent_rows": inconsistent_count,
        "inconsistent_pct": round((inconsistent_count / total_compared * 100), 2) if total_compared > 0 else 0.0,
        "tolerance": tolerance,
        "status": "OK" if inconsistent_count == 0 else "WARNING",
    }
    return result



# Report generation — runs all checks against the actual pipeline tables
def generate_quality_report(spark) -> DataFrame:
    all_results = []

    #Load tables
    municipio_df   = spark.table("bronze.municipio")
    uf_df          = spark.table("bronze.uf")
    directory_df   = spark.table("bronze.municipio_directory")
    students_df    = spark.table("bronze.alunos")
    silver_municipio_df = spark.table("silver.resultado_municipio")

   #Duplicity
    all_results.append(check_duplicates(municipio_df, ["ano", "id_municipio", "rede"], "bronze.municipio"))
    all_results.append(check_duplicates(uf_df, ["ano", "sigla_uf", "rede"], "bronze.uf"))
    all_results.append(check_duplicates(students_df, ["ano", "id_aluno", "id_escola"], "bronze.alunos"))

    #Missing values
    municipal_only_df = silver_municipio_df.filter(col("rede") == "Municipal")

    key_columns_to_check = ["taxa_alfabetizacao_meta_base", "meta_alfabetizacao_2030"]
    missing_value_results = check_missing_values(
        municipal_only_df.select(*[c for c in key_columns_to_check if c in municipal_only_df.columns]),
        "silver.resultado_municipio (rede=Municipal)"
    )
    all_results.extend(missing_value_results)

    all_results.extend(check_missing_values(
        silver_municipio_df.select("municipio_nome"),
        "silver.resultado_municipio (todas as redes)"
    ))

    #Relationship keys
    all_results.append(validate_foreign_key(
        municipio_df, directory_df, "id_municipio",
        "bronze.municipio", "bronze.municipio_directory"
    ))
    all_results.append(validate_foreign_key(
        students_df, directory_df, "id_municipio",
        "bronze.alunos", "bronze.municipio_directory"
    ))

    #Consistency
    if "taxa_alfabetizacao_recalculada" in silver_municipio_df.columns:
        all_results.append(check_consistency(
            silver_municipio_df, silver_municipio_df,
            join_cols=["ano", "id_municipio", "rede"],
            compare_col_a="taxa_alfabetizacao",
            compare_col_b="taxa_alfabetizacao_recalculada",
            tolerance=1.0,
            table_a="silver.resultado_municipio (oficial)",
            table_b="silver.resultado_municipio (recalculada)",
        ))

    #Consolidate into a single Spark DataFrame
    normalized_rows = []
    for r in all_results:
        normalized_rows.append({
            "check": r.get("check", ""),
            "reference": r.get("table") or r.get("child_table") or r.get("table_a", ""),
            "detail": str({k: v for k, v in r.items() if k not in ("check",)}),
            "status": r.get("status", ""),
        })

    report_df = spark.createDataFrame(normalized_rows)
    return report_df


In [0]:
#Call validation
if __name__ == "__main__":
    report_df = generate_quality_report(spark)

    spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
    report_df.write.format("delta").mode("overwrite") \
        .saveAsTable("gold.data_quality_report")

    print("Quality report generated:")
    report_df.groupBy("status").count().show()

    critical_issues = report_df.filter(col("status") == "CRITICAL")
    if critical_issues.count() > 0:
        print("WARNING: the following critical issues have been found:")
        critical_issues.show(truncate=False)
    else:
        print("No critical issues found.")

In [0]:
#Show warnings
display(report_df.filter(col("status") != "OK"))